# 36 — Experience Parsing
**Goal:** Extract company, role, duration, and responsibilities.

## 1. Experience Pattern Recognition

In [ ]:
exp_text = """Google, Mountain View — Senior Data Scientist
Jan 2020 - Present
- Developed NLP pipelines processing 10M+ documents daily
- Led team of 5 ML engineers
- Reduced model latency by 40%

Amazon, Seattle — Data Scientist II
2018 - 2020
- Built recommendation systems
- Improved CTR by 25%
"""

print("Experience entries typically follow:")
print("  Company, Location — Role")
print("  Date range (start - end)")
print("  Bullet points of achievements")

## 2. Experience Parser

In [ ]:
import re

def parse_experience(text):
    """Extract experience entries."""
    entries = []
    lines = text.split("\n")
    current = None
    
    for line in lines:
        ls = line.strip()
        if not ls: continue
        
        # Company line: "Company, Location — Role"
        company_match = re.match(r"^([A-Za-z\s.]+),?\s*([A-Za-z\s]+)?\s*[—\-–]\s*(.+)$", ls)
        if company_match:
            if current: entries.append(current)
            current = {"company": company_match.group(1).strip(), 
                       "role": company_match.group(3).strip(), "duration": "", "bullets": []}
            continue
        
        # Date line
        date_match = re.search(r"\b(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec|\d{4})\s*\d{0,4}\s*-", ls)
        if date_match and current:
            current["duration"] = ls[:40]
            continue
        
        # Bullet point
        bullet = re.sub(r"^[\s•\-*–]+", "", ls)
        if bullet and current and len(bullet) > 10:
            current["bullets"].append(bullet)
    
    if current: entries.append(current)
    return entries

entries = parse_experience(exp_text)
for e in entries:
    print(f"\n  {e['company']:15s} | {e['role']:25s} | {e['duration'][:20]}")
    for b in e['bullets']:
        print(f"    - {b[:50]}...")

## 3. Duration Calculation

In [ ]:
from datetime import datetime

def parse_duration(duration_str):
    """Calculate years from a duration string."""
    years_match = re.search(r"(\d+)\s*(?:\+)?\s*years?", duration_str, re.IGNORECASE)
    if years_match: return int(years_match.group(1))
    
    # Try date range
    range_match = re.findall(r"\b(\d{4})\b", duration_str)
    if len(range_match) >= 2:
        return int(range_match[-1]) - int(range_match[0])
    return 0

for d in ["5+ years", "2020 - Present", "2018 - 2020", "3 years"]:
    print(f"  '{d}' -> {parse_duration(d)} years")

## Summary: Pattern matching extracts structured experience. Duration calc estimates tenure.